## SummaryThis notebook prepares the bi-modal milk flow dataset for downstream analysis.It cleans, merges, and standardizes the raw records needed for lactation-curve and trait-based modeling.

# The effect of bimodal milking on the lactation curve

###Import data from multiple excels at the us blob

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from functools import reduce
import matplotlib.pyplot as plt
import pandas as pd 

In [0]:
#more efficient way to load in the data > get rid of redundant columns in the merging process and make sure information aligns between dataframes.

paths = [f.path for f in dbutils.fs.ls("wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/") if f.path.endswith(".csv")]

needed_cols = [
 'AnimalNumber',
 'LactationNumber',
 'DaysInMilk',
 'Avgmilkflowyesterday1',
 'Avgmilkflowyesterday2',
 'Avgmilkflowyesterday3',
 'DurationYesterdaySession1(mm:ss)',
 'DurationYesterdaySession2(mm:ss)',
 'DurationYesterdaySession3(mm:ss)',
 'Flow30_60Session1Yesterday',
 'Flow30_60Session2Yesterday',
 'Flow30_60Session3Yesterday',
 'LastMilkTime',
 'ReproductionStatus',
 'YieldFirst2Min_Session1Yesterday',
 'YieldFirst2Min_Session2Yesterday',
 'YieldFirst2Min_Session3Yesterday',
 'YieldYesterdaySession1',
 'YieldYesterdaySession2',
 'YieldYesterdaySession3'
 ]

def load_and_fix(path):
    df = spark.read.option("header", "true").option("inferSchema", "false").csv(path)
    df = df.toDF(*[c.replace(" ", "").replace("-", "_").replace(".", "_") for c in df.columns])

    # only keep the required columns
    present = [c for c in needed_cols if c in df.columns]
    missing = [c for c in needed_cols if c not in df.columns]

    df = df.select(present)

    for c in missing:
        df = df.withColumn(c, F.lit(None))

    return df.select(needed_cols)

dfs = [load_and_fix(p) for p in paths]
bi_model_df = reduce(lambda a, b: a.unionByName(b), dfs)

In [0]:
#set schema of dataframe
schema_map = {
    "AnimalNumber": "int",
    "LactationNumber": "int",
    "DaysInMilk": "int",
    "Avgmilkflowyesterday1": "double",
    "Avgmilkflowyesterday2": "double",
    "Avgmilkflowyesterday3": "double",

    # mm:ss — handled later
    "DurationYesterdaySession1(mm:ss)": "string",
    "DurationYesterdaySession2(mm:ss)": "string",
    "DurationYesterdaySession3(mm:ss)": "string",

    "Flow30_60Session1Yesterday": "double",
    "Flow30_60Session2Yesterday": "double",
    "Flow30_60Session3Yesterday": "double",

    "LastMilkTime": "string",   # convert later with try_to_timestamp

    "ReproductionStatus": "string",

    "YieldFirst2Min_Session1Yesterday": "double",
    "YieldFirst2Min_Session2Yesterday": "double",
    "YieldFirst2Min_Session3Yesterday": "double",

    "YieldYesterdaySession1": "double",
    "YieldYesterdaySession2": "double",
    "YieldYesterdaySession3": "double",

}

for colname, dtype in schema_map.items():
    if dtype in ["int", "double", "float"]:
        bi_model_df = bi_model_df.withColumn(colname, F.col(colname).cast(dtype))

    else:  # keep as string
        bi_model_df = bi_model_df.withColumn(colname, F.col(colname))

### Data preprocessing pipeline

In [0]:
# --------------------------------------------
# 1. Clean column names
# --------------------------------------------
needed_cols = [
 'AnimalNumber',
 'LactationNumber',
 'DaysInMilk',
 'Avgmilkflowyesterday1',
 'Avgmilkflowyesterday2',
 'Avgmilkflowyesterday3',
 'DurationYesterdaySession1(mm:ss)',
 'DurationYesterdaySession2(mm:ss)',
 'DurationYesterdaySession3(mm:ss)',
 'Flow30_60Session1Yesterday',
 'Flow30_60Session2Yesterday',
 'Flow30_60Session3Yesterday',
 'LastMilkTime',
 'ReproductionStatus',
 'YieldFirst2Min_Session1Yesterday',
 'YieldFirst2Min_Session2Yesterday',
 'YieldFirst2Min_Session3Yesterday',
 'YieldYesterdaySession1',
 'YieldYesterdaySession2',
 'YieldYesterdaySession3'
 ]

bi_model_df_cleaned = bi_model_df.select(needed_cols)

# --------------------------------------------
# 2. Remove culled cows
# --------------------------------------------
bi_model_df_cleaned = bi_model_df_cleaned.filter(F.col("ReproductionStatus") != "Culled")

# --------------------------------------------
# 3. Drop rows where all 3 flow values = NULL
# --------------------------------------------
bi_model_df_cleaned = bi_model_df_cleaned.filter(
    F.col("Flow30_60Session1Yesterday").isNotNull() |
    F.col("Flow30_60Session2Yesterday").isNotNull() |
    F.col("Flow30_60Session3Yesterday").isNotNull()
)

# --------------------------------------------
# 4. Convert lbs → kg (only for prefixes)
# --------------------------------------------
prefixes = ["Yield", "Avg", "Flow", "Peak"]
factor = 0.45359237

bi_model_df_cleaned = bi_model_df_cleaned.select([
    (F.col(c).try_cast("double") * factor).alias(c)
    if any(c.startswith(p) for p in prefixes)
    else F.col(c)
    for c in bi_model_df_cleaned.columns
])

# --------------------------------------------
# 5. Add LactationId
# --------------------------------------------
bi_model_df_cleaned = bi_model_df_cleaned.withColumn(
    "LactationId",
    F.dense_rank().over(Window.orderBy("AnimalNumber", "LactationNumber"))
)

# --------------------------------------------
# 6. Fix DIM so that the first day where the cow was milked in session 1 is DIM 1
# --------------------------------------------
w = Window.partitionBy("LactationId")

# Create a flag if there is any row in the partition where DaysInMilk == 1 and YieldYesterdaySession1 is NULL
cond = (
    (F.col("YieldYesterdaySession1").isNull()) &
    (F.col("DaysInMilk") == 1)
).cast("int")

bi_model_df_cleaned = bi_model_df_cleaned.withColumn(
    "should_shift",
    F.max(cond).over(w) == 1
)

bi_model_df_cleaned = bi_model_df_cleaned.withColumn(
    "DaysInMilk",
    F.when(F.col("should_shift"), F.col("DaysInMilk") - 1)
     .otherwise(F.col("DaysInMilk"))
)


# --------------------------------------------
# 7. Convert Duration of session to seconds 
# --------------------------------------------
d1 = "DurationYesterdaySession1(mm:ss)"
d2 = "DurationYesterdaySession2(mm:ss)"
d3 = "DurationYesterdaySession3(mm:ss)"

# strict mm:ss pattern: 0-99 minutes, 00-59 seconds 
mmss_pattern = r'^\d{1,2}:[0-5]\d$'

# 1) Drop rows that do not match mm:ss format for ALL three sessions
bi_model_df_cleaned = bi_model_df_cleaned.filter(
    F.col(d1).rlike(mmss_pattern) &
    F.col(d2).rlike(mmss_pattern) &
    F.col(d3).rlike(mmss_pattern)
)

# 2) Efficient vectorized conversion mm:ss -> seconds
def mmss_to_seconds(colname):
    parts = F.split(F.col(colname), ":")
    return parts[0].cast("int") * 60 + parts[1].cast("int")

bi_model_df_cleaned = (
    bi_model_df_cleaned
    .withColumn("DurationSession1_sec", mmss_to_seconds(d1))
    .withColumn("DurationSession2_sec", mmss_to_seconds(d2))
    .withColumn("DurationSession3_sec", mmss_to_seconds(d3))
)

#drop the old duration column that have the time in minutes and the helper column
bi_model_df_cleaned = bi_model_df_cleaned.drop(d1,d2,d3,"should_shift")

# --------------------------------------------
# 8. Outlier filters (all done in single passes), filters based on the Wieland + Sipka 2023 paper. This also ensures that we only have days where the cow had three valid milkings. 
# --------------------------------------------
bi_model_df_cleaned = bi_model_df_cleaned.filter(
    (F.col("YieldYesterdaySession1").between(5, 55)) &
    (F.col("YieldYesterdaySession2").between(5, 55)) &
    (F.col("YieldYesterdaySession3").between(5, 55)) &
    
    (F.col("YieldFirst2Min_Session1Yesterday").between(2, 12)) &
    (F.col("YieldFirst2Min_Session2Yesterday").between(2, 12)) &
    (F.col("YieldFirst2Min_Session3Yesterday").between(2, 12)) &
    
    (F.col("DurationSession1_sec").between(100, 800)) &
    (F.col("DurationSession2_sec").between(100, 800)) &
    (F.col("DurationSession3_sec").between(100, 800))
)

# --------------------------------------------
# 9. Create label if milkflow at 30-60 is lower than 3.1 kg/min. BIM = bimodal milk flow curve 
# --------------------------------------------

bi_model_df_cleaned = (
    bi_model_df_cleaned
    .withColumn("BIM_1", F.when(F.col("Flow30_60Session1Yesterday") < 3.1, F.lit(True)).otherwise(F.lit(False)))
    .withColumn("BIM_2", F.when(F.col("Flow30_60Session2Yesterday") < 3.1, F.lit(True)).otherwise(F.lit(False)))
    .withColumn("BIM_3", F.when(F.col("Flow30_60Session3Yesterday") < 3.1, F.lit(True)).otherwise(F.lit(False)))
    #create a new column for daily milk yield that sums the milkyields of all three sessions
    .withColumn("DailyMilkYield", F.col("YieldYesterdaySession1") + F.col("YieldYesterdaySession2") + F.col("YieldYesterdaySession3"))
)      





In [0]:
display(
    bi_model_df_cleaned.agg(F.min("LastMilkTime").alias("min_LastMilkDate"))
)
display(
    bi_model_df_cleaned.agg(F.max("LastMilkTime").alias("max_LastMilkDate"))
)

In [0]:
display(bi_model_df_cleaned)

## Create label for delayed milk ejection

**Label based on:**
M. Wieland, Anja Sipka,
Comparison of 2 types of milk flow meters for detecting bimodality in dairy cows,
Journal of Dairy Science,
Volume 106, Issue 2,
2023,
Pages 1078-1088,
ISSN 0022-0302,
https://doi.org/10.3168/jds.2022-22437.
(https://www.sciencedirect.com/science/article/pii/S0022030222007135)


In [0]:
#write preprocessed df to the blob 
milk_flow_data = bi_model_df_cleaned.write.mode("overwrite"). \
    parquet(
        "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/milk_flow_data"
    )

In [0]:
dbutils.fs.rm(
    "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/milk_flow_data/_started_1949264512772062973",
    recurse=False
)

In [0]:
#read created dataframe
parquet_file_path = "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/milk_flow_data/"

# Read the Parquet file into a DataFrame > now set to a limit to first make the code working
bi_model_df = spark.read.parquet(parquet_file_path)


In [0]:
bi_model_df.columns

#Descriptive Statistics

In [0]:
#Number of days with observations 
measurement_count = bi_model_df_cleaned.count()
print('Number of days with observation: ', measurement_count)

#Number of lactations
lactations = bi_model_df_cleaned.select('LactationId').distinct().count()
print('Number of lactations: ', lactations )

#Average number of observations per lactation 
print('Average number of observation days per lactation: ', measurement_count/lactations)

#Average lactationlength 
avg_lactationlength = bi_model_df_cleaned.select('LactationId', 'DaysInMilk').groupBy('LactationId').agg(F.max('DaysInMilk').alias('LactationLength')).agg(F.avg('LactationLength')).collect()
print('Average lactationlength: ', avg_lactationlength[0][0])

#Number of milkings with a bimodal milk flow curve during at least one of the milkings
BMI_count = bi_model_df_cleaned.filter((bi_model_df_cleaned.BIM_1 == True) | (bi_model_df_cleaned.BIM_2 == True) | (bi_model_df_cleaned.BIM_3 == True)).count()
print('Number of milkings with bimodal milk flow curve: ', BMI_count)



In [0]:
display(bi_model_df_cleaned.filter(
    F.col("Flow30_60Session1Yesterday").isNull() |
    F.col("Flow30_60Session2Yesterday").isNull() |
    F.col("Flow30_60Session3Yesterday").isNull()
))

In [0]:
#Number of milkings with observations 
total_non_null = (
    bi_model_df_cleaned
    .select(
        F.sum(F.col("Flow30_60Session1Yesterday").isNotNull().cast("int")) +
        F.sum(F.col("Flow30_60Session2Yesterday").isNotNull().cast("int")) +
        F.sum(F.col("Flow30_60Session3Yesterday").isNotNull().cast("int"))
    ).collect()[0][0]
)
print('Number of milkings with observation: ', total_non_null)

#Number of milkings with bimodal milk flow curve 
total_bim_sessions = bi_model_df_cleaned.select(
    (F.sum(F.col("BIM_1").cast("int")) +
     F.sum(F.col("BIM_2").cast("int")) +
     F.sum(F.col("BIM_3").cast("int"))).alias("total_bim")
).collect()[0]["total_bim"]

print("Total bimodal sessions:", total_bim_sessions)

#Proportion of milkings with bimodal milk flow curve 
print('Proportion of milkings with bimodal milk flow curve: ', total_bim_sessions/total_non_null)

### Visualizations 

In [0]:
# Convert Spark DF to pandas-on-Spark DF
subset = bi_model_df_cleaned.limit(500).toPandas()

# Distribution of DailyMilkYield
subset['DailyMilkYield'].plot.hist(bins=100, figsize=(10,5), title='Distribution of Daily Milk Yield')
plt.show()

# Distribution of milk flow sessions
for col in ['Flow30_60Session1Yesterday', 'Flow30_60Session2Yesterday', 'Flow30_60Session3Yesterday']:
    subset[col].plot.hist(bins=100, figsize=(10,5), title=f'Distribution of {col}')
    plt.show()

#Code graveyard

In [0]:
# old way to load in the data 
# order of the columns is not consistent over the excel files. This causes misassignment of values to the wrong column. Solve by reading one by one and fix the column names before

# ---- folder path ----
folder = "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/"

# list CSV files (Databricks helper)
files = [f.path for f in dbutils.fs.ls(folder) if f.path.lower().endswith(".csv")]

# helper: normalize column names exactly as used later
def normalize_cols(cols):
    return [c.replace(" ", "").replace("-", "_").replace(".", "_") for c in cols]

# ---- Step 1: build the superset of all headers  ----
all_cols = set()
for p in files:
    # read only the first line (fast), using dbutils.fs.head
    header_line = dbutils.fs.head(p, 10000).splitlines()[0]
    # header_line is e.g. "AnimalNumber,DaysInMilk,ReproductionStatus,..."
    cols = [c.strip().strip('"') for c in header_line.split(",")]
    cols = normalize_cols(cols)
    all_cols.update(cols)

# define the canonical expected column order:
# put important columns first (custom order), then the rest sorted
important = ["AnimalNumber", "LactationNumber", "DaysInMilk"] 
remaining = sorted([c for c in all_cols if c not in important])
expected_cols = [c for c in important if c in all_cols] + remaining

# ---- Step 2: read each file, normalize headers, and union by name ----
df_all = None
for p in files:
    df_tmp = spark.read.option("header", "true").option("inferSchema", "false").csv(p)
    normalized = normalize_cols(df_tmp.columns)
    if normalized != list(df_tmp.columns):
        df_tmp = df_tmp.toDF(*normalized)
    
    # Use set for case-insensitive check
    existing_cols = set([c.lower() for c in df_tmp.columns])
    missing = [c for c in expected_cols if c.lower() not in existing_cols]
    for c in missing:
        df_tmp = df_tmp.withColumn(c, F.lit(None).cast("string"))
    
    # Select only unique columns in canonical order
    unique_cols = []
    seen = set()
    for c in expected_cols:
        cl = c.lower()
        if cl not in seen:
            unique_cols.append(c)
            seen.add(cl)
    df_tmp = df_tmp.select(*unique_cols)
    
    if df_all is None:
        df_all = df_tmp
    else:
        df_all = df_all.unionByName(df_tmp, allowMissingColumns=True)

df = df_all

